## Assignment 6

<br>

#### Exercise 6.1

Consider the dataset from `data_banknote_authentication.csv`.

1) Read data into a pandas dataframe.

2) Pick the column named "class" as target variable `y` and all other columns as feature variables `X`.

3) Split the data into training and testing sets with 80/20 ratio and `random_state=20`.

4) Use support vector classifier with linear kernel to fit to the training data.

5) Predict on the testing data and compute the confusion matrix and classification report.

6) Repeat steps 3 and 4 for the radial basis function kernel.

7) Compare the two SVM models in your own words.

<br>

#### Exercise 6.2

This exercise is related to exercise 5.2 of the previous week. Consider the data from CSV file `weight-height.csv`.

1) Read data into a pandas dataframe.

2) Pick the target variable `y` as weight in kilograms, and the feature variable `X` as height in centimeters.

3) Split the data into training and testing sets with 80/20 ratio.

4) Scale the training and testing data using normalization and standardization.

4) Fit a KNN regression model with `k=5` to the training data without scaling, predict on unscaled testing data and compute the $R^2$ value.

6) Repeat step 4 for normalized data.

7) Repeat step 4 for standardize data.

8) Compare the models in terms of their $R^2$ value.

## Exercise 6.1 – Banknote authentication: SVM classifier

### 1) Read `data_banknote_authentication.csv`

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, classification_report

df = pd.read_csv('data_banknote_authentication.csv')

print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)

df.head()

### 2) Target `y` = `class`, features `X` = all remaining columns

In [ ]:
X = df.drop('class', axis=1)
y = df['class']

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nClass balance:")
print(y.value_counts())

### 3) Train/test split (80/20, `random_state=20`)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=20
)

print("Training set:", X_train.shape)
print("Testing set: ", X_test.shape)

### 4) Support vector classifier – linear kernel

In [ ]:
svc_linear = SVC(kernel='linear')
svc_linear.fit(X_train, y_train)

### 5) Predict, confusion matrix and classification report (linear kernel)

In [ ]:
y_pred_linear = svc_linear.predict(X_test)

cm_linear = confusion_matrix(y_test, y_pred_linear)
print("Confusion matrix (linear kernel):")
print(cm_linear)
print("\nClassification report (linear kernel):")
print(classification_report(y_test, y_pred_linear))

plt.figure(figsize=(4, 3.5))
sns.heatmap(cm_linear, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Linear Kernel SVM')
plt.tight_layout()
plt.show()

### 6) Repeat for the radial basis function (RBF) kernel

In [ ]:
# Same 80/20 split as before is reused (already created with random_state=20)
svc_rbf = SVC(kernel='rbf')
svc_rbf.fit(X_train, y_train)

y_pred_rbf = svc_rbf.predict(X_test)

cm_rbf = confusion_matrix(y_test, y_pred_rbf)
print("Confusion matrix (RBF kernel):")
print(cm_rbf)
print("\nClassification report (RBF kernel):")
print(classification_report(y_test, y_pred_rbf))

plt.figure(figsize=(4, 3.5))
sns.heatmap(cm_rbf, annot=True, fmt='d', cmap='Greens')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - RBF Kernel SVM')
plt.tight_layout()
plt.show()

### 7) Comparison of the two SVM models

Both models perform extremely well on this dataset, which makes sense because the banknote features (variance, skewness, curtosis, entropy of the wavelet-transformed image) separate the two classes quite cleanly.

The **linear kernel** SVM already achieves very high accuracy (around 99%), with only a couple of misclassifications. This indicates that a straight hyperplane in the original 4-dimensional feature space is almost enough to separate genuine from forged banknotes.

The **RBF kernel** SVM performs even better, achieving (or coming very close to) perfect classification on the test set. The RBF kernel can model non-linear, curved decision boundaries, so it can capture the small amount of non-linear structure in the data that the linear kernel misses. In this particular case the improvement is small in absolute terms, simply because the linear model was already close to ceiling, but the RBF model is still the more flexible — and here, slightly more accurate — choice.

In general, the trade-off is: the linear kernel is simpler, faster to train, and easier to interpret (the coefficients directly tell you how each feature contributes), while the RBF kernel is more flexible and can fit more complex boundaries, at the cost of being a "black box" and having extra hyperparameters (like `C` and `gamma`) that may need tuning to avoid overfitting. For this dataset, since the classes are nearly linearly separable, both kernels work very well, but RBF has a slight edge.

## Exercise 6.2 – Height/Weight data: KNN regression with different scalings

### 1) Read data into a pandas dataframe

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score

df_wh = pd.read_csv('weight-height.csv')
df_wh.head()

### 2) Target `y` = weight in kg, feature `X` = height in cm

The raw data is in inches and pounds, so we convert it first.

In [ ]:
df_wh['Height_cm'] = df_wh['Height'] * 2.54        # inches -> cm
df_wh['Weight_kg'] = df_wh['Weight'] * 0.453592      # pounds -> kg

X = df_wh[['Height_cm']]
y = df_wh['Weight_kg']

print(X.head())
print(y.head())

### 3) Train/test split (80/20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set:", X_train.shape)
print("Testing set: ", X_test.shape)

### 4) Scale the data: normalization and standardization

- **Normalization** (`MinMaxScaler`) rescales values to the range [0, 1].
- **Standardization** (`StandardScaler`) rescales values to have mean 0 and standard deviation 1.

Each scaler is fit only on the training data, then applied (`transform`) to the test data, to avoid leaking test-set information into the scaling parameters.

In [ ]:
# Normalization
minmax_scaler = MinMaxScaler()
X_train_norm = minmax_scaler.fit_transform(X_train)
X_test_norm = minmax_scaler.transform(X_test)

# Standardization
std_scaler = StandardScaler()
X_train_std = std_scaler.fit_transform(X_train)
X_test_std = std_scaler.transform(X_test)

print("Normalized training sample:\n", X_train_norm[:5])
print("\nStandardized training sample:\n", X_train_std[:5])

### 5) KNN regression (k=5) without scaling

In [ ]:
knn_unscaled = KNeighborsRegressor(n_neighbors=5)
knn_unscaled.fit(X_train, y_train)

y_pred_unscaled = knn_unscaled.predict(X_test)
r2_unscaled = r2_score(y_test, y_pred_unscaled)

print(f"R-squared (unscaled features): {r2_unscaled:.4f}")

### 6) Repeat for normalized data

In [ ]:
knn_norm = KNeighborsRegressor(n_neighbors=5)
knn_norm.fit(X_train_norm, y_train)

y_pred_norm = knn_norm.predict(X_test_norm)
r2_norm = r2_score(y_test, y_pred_norm)

print(f"R-squared (normalized features): {r2_norm:.4f}")

### 7) Repeat for standardized data

In [ ]:
knn_std = KNeighborsRegressor(n_neighbors=5)
knn_std.fit(X_train_std, y_train)

y_pred_std = knn_std.predict(X_test_std)
r2_std = r2_score(y_test, y_pred_std)

print(f"R-squared (standardized features): {r2_std:.4f}")

### 8) Comparison of the models

In [ ]:
comparison = pd.DataFrame({
    'Scaling': ['None (unscaled)', 'Normalization (MinMax)', 'Standardization (StandardScaler)'],
    'R2': [r2_unscaled, r2_norm, r2_std]
})
comparison

**Discussion:**

All three $R^2$ values come out essentially identical. This is expected here, because there is only **one** feature (`Height_cm`). Both `MinMaxScaler` and `StandardScaler` apply a purely linear, monotonic transformation to that single feature — they shift and stretch the values but never change their *relative order or relative distances* in a way that would change which points are nearest neighbours to each other. Since KNN with a single feature only cares about the ranking/relative distance between points, rescaling that one feature doesn't change which 5 neighbours are picked for any test point, so the predictions (and therefore $R^2$) stay the same.

Scaling becomes important for KNN when there are **multiple features on different scales** (e.g. height in cm and income in dollars) — in that case, an unscaled feature with a larger numeric range would dominate the distance calculation, and normalization/standardization is needed to give each feature a fair contribution. With only one feature, as in this exercise, scaling has no effect on the model's performance.